# face-cloak: full experiment run on Kaggle (GPU)

> **Keep this notebook private.** The results contain real LFW faces and deepfake swaps of them (qualitative grids,
> cloaked PNGs). Never make the notebook or its output public. Do not publish `results*.zip`; it is for the report
> authors only. Before committing this file, make sure its outputs are cleared.

Runs the whole evaluation of the project: weights download, LFW + manifest (150 sources / 30 targets / 3 targets per
source), calibration of the judges, then the experiment configs (`main`, `ablation_eps`, `ablation_steps`), figures and
tables, and finally packs everything into one zip in `/kaggle/working` (the **Output** tab).

**Before you run:** right panel -> **Settings** -> **Accelerator: GPU T4 x2** or **GPU P100** (the code uses a single GPU either way), and **Internet: On** (needs a phone-verified Kaggle account). Then use
**Save Version -> Save & Run All (Commit)** so it keeps running in the background without your browser.
Step-by-step guide: `KAGGLE_RUN.md` in the repo.

**Runtime (estimate, one T4):** about 1.3 h setup + calibration, `main` about 9-12 h, `ablation_eps` about 2.5-3 h,
`ablation_steps` about 1-1.5 h (total about 14-17 h). That is more than one 12 h session, so the notebook is built to stop cleanly at
`TIME_BUDGET_H`, pack what is done, and **resume** in the next session (`RESUME_FROM`). Splitting across sessions is
described in the config cell below and in `KAGGLE_RUN.md`.

**Resumable and re-runnable:** every cell can be run again. Finished work is never redone (cloaked images and CSV rows
are cached and checked by hash); downloads are skipped if the files are already there and pass their checksum.

**Cells:** 1 config + helpers, 2 setup (clone, pip, GPU check), 3 weights + SimSwap, 4 data + manifest,
5 calibration, 6 experiments, 7 figures and tables, 8 package + summary.

In [ ]:
# 1) Config + helpers. Edit only the CONFIG block. Safe to re-run (note: re-running restarts the TIME_BUDGET clock).
import csv, glob, hashlib, json, os, queue, re, shutil, signal, subprocess, sys, threading, time, zipfile
from datetime import datetime, timezone
from pathlib import Path

# ============================== CONFIG ==============================
SHARD = "0/1"            # "i/N": this session handles sources i, i+N, ... (see KAGGLE_RUN.md before changing)
RESUME_FROM = None       # e.g. "/kaggle/input/facecloak-partial/results.zip" (a .zip or an extracted folder)
REPO_URL = "https://github.com/J4KE-B/face-cloak"
REPO_REF = "main"        # branch, tag or full commit SHA; pin a SHA for the real run so the code cannot change
RUN_CONFIGS = ["main", "ablation_eps", "ablation_steps"]   # priority order
# Estimate (one T4): all three configs ~ 14-17 h + ~1.3 h setup/calibration, i.e. MORE than one 12 h session. Options:
#   RECOMMENDED  two sessions, same notebook, each does half of the sources of every config (can run at the same time):
#       session A:  SHARD = "0/2"        session B:  SHARD = "1/2"       (each ~ 7-9 h)
#       (B can reuse A's thresholds: CALIBRATION_JSON = ".../calibration.json"; otherwise it recomputes the same ones)
#   or by config:
#       session 1:  RUN_CONFIGS = ["main"]                            SHARD = "0/1"   (~ 10-13 h, may hit the budget)
#       session 2:  RUN_CONFIGS = ["ablation_eps", "ablation_steps"]  SHARD = "0/1"   (~ 4-5 h)
#   If a session stops at the budget, run again with RESUME_FROM = <its zip>: finished work is skipped.
TIME_BUDGET_H = 11.0     # Kaggle kills sessions at 12 h. After this budget: the running config is stopped cleanly
                         # (resumable), no new config is started, and results are still packaged.
CALIBRATION_JSON = None  # optional: path to an existing results/calibration.json (skips the slow CPU calibration)
EXPECTED_MANIFEST_FP = "ad5d8ff2596cb5905d5bd549ee037d7610a45ba9bbfaade0bdd93f94c791b2cd"
                         # fingerprint of the local manifest (ids, identities, image paths, pairs); None = don't check
ALLOW_MANIFEST_MISMATCH = False
REQUIRE_ORT_CUDA = True  # stop early if onnxruntime cannot use the GPU (swaps would be ~10x slower)
THREADS = 4              # Kaggle has 4 CPU cores
SNAPSHOT_EVERY_MIN = 60  # re-write the zip every hour while running (safety net if the session dies)
# ====================================================================

T_START = time.time()
DEADLINE = T_START + TIME_BUDGET_H * 3600
m = re.fullmatch(r"(\d+)/(\d+)", SHARD)
assert m and int(m[1]) < int(m[2]), f"SHARD must look like '0/1' or '1/2', got {SHARD!r}"
SHARD_I, SHARD_N = int(m[1]), int(m[2])

SCRATCH = Path("/tmp/facecloak")              # big/temporary stuff stays out of /kaggle/working
REPO = SCRATCH / "face-cloak"
WEIGHTS = SCRATCH / "weights"
LFW = SCRATCH / "lfw"
RESULTS = SCRATCH / "results"                 # raw CSVs, caches, figs ... packed into the zip at the end
LOGS = RESULTS / "logs"
OUT = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else SCRATCH / "out"
ZIP_PATH = OUT / ("results.zip" if SHARD == "0/1" else f"results_shard{SHARD_I}of{SHARD_N}.zip")
for d in (SCRATCH, WEIGHTS, LFW, RESULTS, LOGS, OUT):
    d.mkdir(parents=True, exist_ok=True)

# NOTE: FACECLOAK_DATA is left at its default (<repo>/data) on purpose: configs/*.yaml refer to data/manifest.json
# relative to the repo, so the manifest must live there. Only the big LFW download goes to scratch.
os.environ.update({
    "FACECLOAK_WEIGHTS": str(WEIGHTS), "FACECLOAK_LFW": str(LFW), "FACECLOAK_RESULTS": str(RESULTS),
    "FACECLOAK_THREADS": str(THREADS), "FACECLOAK_ORT_GPU": "1", "PYTHONUNBUFFERED": "1", "TQDM_MININTERVAL": "120",
    "PYTHONPATH": os.pathsep.join(filter(None, [str(REPO), os.environ.get("PYTHONPATH", "")])),
})
PY = sys.executable

if RESUME_FROM and not Path(RESUME_FROM).exists():
    shown = sorted(glob.glob("/kaggle/input/*/*"))[:20]
    raise SystemExit(f"RESUME_FROM {RESUME_FROM!r} does not exist. Found under /kaggle/input: {shown}")


def sh(cmd, *, log=None, deadline=None, snapshot=False, check=True, cwd=None):
    """Run a command, stream its output live (and into a log file). Returns (returncode, hit_deadline).
    At `deadline` the child gets SIGINT (clean stop, work is resumable), SIGKILL after 120 s."""
    cmd = [str(c) for c in cmd]
    print("$", " ".join(cmd), flush=True)
    logf = open(log, "a") if log else None
    p = subprocess.Popen(cmd, cwd=str(cwd or REPO), env=os.environ.copy(), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, errors="replace", bufsize=1, start_new_session=True)
    q = queue.Queue()

    def reader():
        for ln in p.stdout:
            q.put(ln)
        q.put(None)

    threading.Thread(target=reader, daemon=True).start()
    hit, t_int, last_snap = False, 0.0, time.time()
    while True:
        try:
            ln = q.get(timeout=5)
        except queue.Empty:
            ln = ""
        else:
            if ln is None:
                break
        if ln:
            print(ln, end="", flush=True)
            if logf:
                logf.write(ln); logf.flush()
        now = time.time()
        if deadline and now > deadline and not hit:
            hit, t_int = True, now
            print(f"\n*** TIME BUDGET REACHED: stopping `{cmd[1] if len(cmd) > 1 else cmd[0]}` cleanly (resumable) ***", flush=True)
            os.killpg(p.pid, signal.SIGINT)
        if hit and now - t_int > 120 and p.poll() is None:
            os.killpg(p.pid, signal.SIGKILL)
        if snapshot and now - last_snap > SNAPSHOT_EVERY_MIN * 60:
            package(partial=True); last_snap = time.time()
    rc = p.wait()
    if logf:
        logf.close()
    if check and rc != 0 and not hit:
        raise RuntimeError(f"command failed with exit code {rc}: {' '.join(cmd)}")
    return rc, hit


def count_rows(path):
    p = Path(path)
    if not p.is_file():
        return 0
    with open(p, newline="") as f:
        return max(0, sum(1 for _ in csv.reader(f)) - 1)


def find_in_resume(name):
    """Bytes of file `name` inside RESUME_FROM (zip or folder), or None."""
    if not RESUME_FROM:
        return None
    p = Path(RESUME_FROM)
    if p.is_dir():
        hits = sorted(p.rglob(name))
        return hits[0].read_bytes() if hits else None
    if zipfile.is_zipfile(p):
        with zipfile.ZipFile(p) as z:
            for n in z.namelist():
                if Path(n).name == name:
                    return z.read(n)
    return None


def package(partial=False):
    """Zip RESULTS (raw CSVs/meta, calibration, manifest, cloaked-PNG cache, figs, tables, logs) -> ZIP_PATH, atomically."""
    info = {"state": "partial" if partial else "final", "written": datetime.now(timezone.utc).isoformat(timespec="seconds"),
            "shard": SHARD, "run_configs": RUN_CONFIGS, "repo_ref": REPO_REF, "resume_from": RESUME_FROM,
            "elapsed_h": round((time.time() - T_START) / 3600, 2)}
    try:
        info["repo_commit"] = subprocess.run(["git", "-C", str(REPO), "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
    except Exception:
        pass
    (RESULTS / "run_info.json").write_text(json.dumps(info, indent=1))
    (RESULTS / "README_PRIVATE.txt").write_text(
        "face-cloak Kaggle results. PRIVATE: figs/qualitative* and cache/cloaked contain real LFW faces and deepfake swaps.\n"
        "Merge with: python scripts/merge_results.py <this zip> [other zips] --out results/raw --cache results/cache/cloaked\n")
    tmp = ZIP_PATH.with_name(ZIP_PATH.name + ".part")
    n = 0
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=6) as z:
        def add(path):
            nonlocal n
            if path.is_file() and path.suffix not in (".npz", ".part"):
                kind = zipfile.ZIP_STORED if path.suffix == ".png" else zipfile.ZIP_DEFLATED
                z.write(path, path.relative_to(RESULTS).as_posix(), compress_type=kind)
                n += 1
        for sub in ("raw", "cache/cloaked", "figs", "tables", "web", "logs"):
            for f in sorted((RESULTS / sub).rglob("*")) if (RESULTS / sub).is_dir() else []:
                add(f)
        for name in ("calibration.json", "manifest.json", "run_info.json", "README_PRIVATE.txt"):
            add(RESULTS / name)
    os.replace(tmp, ZIP_PATH)
    print(f"[package] {'snapshot' if partial else 'final'}: {n} files -> {ZIP_PATH} ({ZIP_PATH.stat().st_size / 1e6:.1f} MB)", flush=True)


print(f"shard {SHARD}  configs {RUN_CONFIGS}  budget {TIME_BUDGET_H} h  resume_from={RESUME_FROM}")
print(f"repo {REPO}\nweights {WEIGHTS}\nlfw {LFW}\nresults {RESULTS}\nzip -> {ZIP_PATH}")

In [ ]:
# 2) Setup: clone the repo at REPO_REF, install dependencies (torch/CUDA are preinstalled on Kaggle: do NOT reinstall torch)
if not (REPO / ".git").is_dir():
    REPO.mkdir(parents=True, exist_ok=True)
    sh(["git", "init", "-q", REPO], cwd=SCRATCH)
    sh(["git", "remote", "add", "origin", REPO_URL], cwd=REPO)
sh(["git", "fetch", "-q", "--depth", "1", "origin", REPO_REF], cwd=REPO)   # works for branches, tags and full SHAs
sh(["git", "checkout", "-q", "--force", "FETCH_HEAD"], cwd=REPO)
sh(["git", "log", "-1", "--format=repo at %H  %s"], cwd=REPO)

sh([PY, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.txt"])
# the project metadata says python <3.12; Kaggle's interpreter may differ, the code itself does not need that pin
sh([PY, "-m", "pip", "install", "-q", "--no-deps", "--ignore-requires-python", "-e", "."])

# GPU / library report (in a subprocess so this notebook process does not hold a CUDA context)
sh([PY, "-c", (
    "import torch, onnxruntime as ort, numpy, sys;"
    "print('python', sys.version.split()[0], '| numpy', numpy.__version__);"
    "print('torch', torch.__version__, '| cuda available', torch.cuda.is_available(), '| torch cuda', torch.version.cuda);"
    "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None);"
    "print('onnxruntime', ort.__version__, ort.get_available_providers());"
    "sys.exit(0 if torch.cuda.is_available() else 2)")], check=False)
r = subprocess.run([PY, "-c", "import torch,sys; sys.exit(0 if torch.cuda.is_available() else 1)"], capture_output=True)
if r.returncode != 0:
    raise SystemExit("No CUDA GPU visible to torch. Settings -> Accelerator -> GPU T4 x1 (then the session restarts), and re-run.")
sh(["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"], check=False)
sh(["df", "-h", "/tmp", "/kaggle/working"], check=False)

In [ ]:
# 3) Weights (sha256-pinned downloads), SimSwap repo at the pinned commit, and a GPU check for onnxruntime
sh([PY, "-m", "facecloak.weights", "--fetch"])                                                    # inswapper, buffalo_l/s, antelopev2, facenet
sh([PY, "-m", "facecloak.weights", "--fetch", "simswap_checkpoints", "simswap_arcface", "--force-kaggle-only"])
sh([PY, "-m", "facecloak.weights", "--check"])                                                    # exit code != 0 on any mismatch

# SimSwap code: the repo has no clone helper, so do it here. Its code is imported and executed by facecloak.swap,
# which refuses anything but the pinned commit. Pin is read from the repo so the two can never drift apart.
SIMSWAP_COMMIT = re.search(r'SIMSWAP_COMMIT\s*=\s*"([0-9a-f]{40})"', (REPO / "facecloak" / "swap.py").read_text())[1]
SIMSWAP_REPO = WEIGHTS / "simswap_repo"


def head(repo):
    r = subprocess.run(["git", "-C", str(repo), "rev-parse", "HEAD"], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ""


if head(SIMSWAP_REPO) != SIMSWAP_COMMIT:
    if not (SIMSWAP_REPO / ".git").is_dir():
        SIMSWAP_REPO.mkdir(parents=True, exist_ok=True)
        sh(["git", "init", "-q", SIMSWAP_REPO], cwd=SCRATCH)
        sh(["git", "remote", "add", "origin", "https://github.com/neuralchen/SimSwap"], cwd=SIMSWAP_REPO)
    sh(["git", "fetch", "-q", "--depth", "1", "origin", SIMSWAP_COMMIT], cwd=SIMSWAP_REPO)
    sh(["git", "checkout", "-q", "--force", SIMSWAP_COMMIT], cwd=SIMSWAP_REPO)
assert head(SIMSWAP_REPO) == SIMSWAP_COMMIT, f"SimSwap repo is at {head(SIMSWAP_REPO)}, expected {SIMSWAP_COMMIT}"
assert (SIMSWAP_REPO / "models" / "fs_networks.py").is_file(), "SimSwap clone is missing models/fs_networks.py"
print("SimSwap repo OK at", SIMSWAP_COMMIT)

# Can onnxruntime really use the GPU? (same import order as the real scripts: torch first, then onnxruntime)
ORT_TEST = ("import torch, onnxruntime as ort, sys;"
            "s = ort.InferenceSession(sys.argv[1], providers=['CUDAExecutionProvider', 'CPUExecutionProvider']);"
            "print('ORT session providers:', s.get_providers()); sys.exit(0 if s.get_providers()[0] == 'CUDAExecutionProvider' else 3)")
r50 = WEIGHTS / "models" / "buffalo_l" / "w600k_r50.onnx"
ok = sh([PY, "-c", ORT_TEST, r50], check=False)[0] == 0
if not ok:   # fallback: expose the pip-installed NVIDIA libs (cuDNN/cuBLAS) to the dynamic loader
    import site
    libs = [d for sp in site.getsitepackages() for d in glob.glob(f"{sp}/nvidia/*/lib")]
    os.environ["LD_LIBRARY_PATH"] = os.pathsep.join(libs + [os.environ.get("LD_LIBRARY_PATH", "")])
    ok = sh([PY, "-c", ORT_TEST, r50], check=False)[0] == 0
    print("onnxruntime GPU after LD_LIBRARY_PATH fallback:", ok)
if not ok:
    msg = ("onnxruntime cannot use CUDA (it would silently fall back to CPU: inswapper swaps ~10x slower). Try: "
           "pip uninstall -y onnxruntime onnxruntime-gpu && pip install onnxruntime-gpu, then re-run this cell.")
    if REQUIRE_ORT_CUDA:
        raise SystemExit(msg)
    print("WARNING:", msg)

In [ ]:
# 4) Data: LFW (figshare, sha256-checked) + evaluation manifest (150 sources / 30 targets / 3 targets per source, seed 0)
MANIFEST = REPO / "data" / "manifest.json"        # where configs/*.yaml expect it (relative to the repo root)
if not MANIFEST.is_file():
    MANIFEST.parent.mkdir(parents=True, exist_ok=True)
    sh([PY, "scripts/prepare_data.py", "--n-ids", "150", "--n-targets", "30", "--targets-per-src", "3",
        "--seed", "0", "--out", MANIFEST], log=LOGS / "prepare_data.log")
else:
    print("manifest already present:", MANIFEST)

man = json.loads(MANIFEST.read_text())
print(f"sources={len(man['sources'])} targets={len(man['targets'])} pairs={len(man['pairs'])}")
print("first source ids:", [s["id"] for s in man["sources"][:5]], "| first identities:", [s["identity"] for s in man["sources"][:3]])
print("first pairs:", man["pairs"][:3])
assert (len(man["sources"]), len(man["targets"]), len(man["pairs"])) == (150, 30, 450), "unexpected manifest size"
print("manifest file sha256:", hashlib.sha256(MANIFEST.read_bytes()).hexdigest())

# Structural fingerprint: independent of float noise in boxes/landmarks (which can differ slightly between onnxruntime
# builds), but changes if any identity, image or pair differs. It must match the local manifest, otherwise results from
# this run and from the laptop cannot be merged meaningfully.
fp_data = [[[s["id"], s["identity"], s["src_path"], s["ref_path"]] for s in man["sources"]],
           [[t["id"], t["identity"], t["path"]] for t in man["targets"]], man["pairs"]]
fp = hashlib.sha256(json.dumps(fp_data, separators=(",", ":")).encode()).hexdigest()
print("manifest structural fingerprint:", fp)
if EXPECTED_MANIFEST_FP and fp != EXPECTED_MANIFEST_FP:
    msg = f"manifest fingerprint {fp} != expected {EXPECTED_MANIFEST_FP} (different LFW files / selection than the local manifest)"
    if not ALLOW_MANIFEST_MISMATCH:
        raise SystemExit(msg + ". Set ALLOW_MANIFEST_MISMATCH = True only if you will NOT merge with local results.")
    print("WARNING:", msg)
else:
    print("manifest matches the expected fingerprint" if EXPECTED_MANIFEST_FP else "(no expected fingerprint set)")
shutil.copyfile(MANIFEST, RESULTS / "manifest.json")      # shipped back inside the zip

In [ ]:
# 5) Calibration of the judges on LFW pairs (FAR=1% thresholds -> results/calibration.json).
# scripts/calibrate.py runs on CPU (ORT CPUExecutionProvider is hard-wired there): expect roughly 45-90 min on 4 cores.
# It is skipped if a valid calibration.json already exists, was given as CALIBRATION_JSON, or is inside RESUME_FROM
# (then every session uses the very same thresholds).
CALIB = RESULTS / "calibration.json"
JUDGES = ("glintr100", "w600k_mbf", "w600k_r50")


def calib_ok(p):
    try:
        return all(j in json.loads(Path(p).read_text()) for j in JUDGES)
    except Exception:
        return False


if not calib_ok(CALIB):
    if CALIBRATION_JSON and Path(CALIBRATION_JSON).is_file():
        shutil.copyfile(CALIBRATION_JSON, CALIB); print("using CALIBRATION_JSON", CALIBRATION_JSON)
    elif (blob := find_in_resume("calibration.json")) is not None:
        CALIB.write_bytes(blob); print("using calibration.json found in RESUME_FROM")
if not calib_ok(CALIB):
    CALIB.unlink(missing_ok=True)
    sh([PY, "scripts/calibrate.py", "--threads", THREADS, "--out", CALIB], log=LOGS / "calibrate.log")
assert calib_ok(CALIB), "calibration.json is missing a judge"

cal = json.loads(CALIB.read_text())
print(f"{'judge':12s} {'10-fold acc':>12s} {'tau@FAR1%':>10s} {'TAR@FAR1%':>10s} {'EER':>7s}  pairs(g/i)")
for j in JUDGES:
    c = cal[j]
    print(f"{j:12s} {c['acc_10fold_mean']:12.4f} {c['tau_far1']:10.4f} {c['tar_far1']:10.4f} {c['eer']:7.4f}  {c['n_genuine']}/{c['n_impostor']}")
for j in ("glintr100", "w600k_r50"):
    assert cal[j]["acc_10fold_mean"] >= 0.99, f"{j}: LFW 10-fold accuracy {cal[j]['acc_10fold_mean']:.4f} < 0.99, something is wrong (detection/alignment?)"
print("calibration sanity check passed")

In [ ]:
# 6) Experiments. One config at a time, in priority order, until TIME_BUDGET_H. A config that crashes is logged and the
# next one still runs. Everything is resumable: re-running this cell (or the whole notebook, or a new session with
# RESUME_FROM) only does what is missing. The zip is refreshed after every config and hourly while one is running.
MIN_LAUNCH_S = 15 * 60          # do not start a new config with less than 15 min of budget left
status = {}
for c in RUN_CONFIGS:
    left = DEADLINE - time.time()
    if left < MIN_LAUNCH_S:
        status[c] = "skipped (time budget)"; print(f"\n=== {c}: skipped, only {left / 60:.0f} min of budget left ===")
        continue
    print(f"\n=== {c}  (shard {SHARD}, {left / 3600:.1f} h of budget left) ===", flush=True)
    cmd = [PY, "scripts/run_experiments.py", "--config", f"configs/{c}.yaml", "--shard", SHARD,
           "--out", RESULTS / "raw", "--cache", RESULTS / "cache" / "cloaked"]
    if RESUME_FROM:
        cmd += ["--resume-from", RESUME_FROM]
    t0 = time.time()
    try:
        rc, hit = sh(cmd, log=LOGS / f"run_{c}_shard{SHARD_I}of{SHARD_N}.log", deadline=DEADLINE, snapshot=True, check=False)
        status[c] = ("stopped at time budget (resumable)" if hit else "done" if rc == 0 else f"FAILED (exit {rc}), see logs")
    except Exception as e:      # keep going with the next config
        status[c] = f"FAILED ({type(e).__name__}: {e})"
    print(f"=== {c}: {status[c]} after {(time.time() - t0) / 60:.1f} min ===", flush=True)
    sh([PY, "scripts/remaining_work.py", "--config", f"configs/{c}.yaml", "--results", RESULTS / "raw"], check=False)
    package(partial=True)
print("\nSUMMARY:", json.dumps(status, indent=1))
if any(v.startswith("FAILED") for v in status.values()):
    print("At least one config FAILED: read its log in the output above or in results/logs inside the zip.")

In [ ]:
# 7) Figures + tables from whatever is finished so far. Each step is allowed to fail without stopping the notebook.
# PRIVATE: the qualitative grid shows real LFW faces next to deepfakes of them. It is created LAST so it is not copied
# into web/ by export_web_results.py, and it only ever goes into the zip (report authors only). Never publish it.
FIG_STATUS = {}
n_swap = sum(count_rows(f) for f in (RESULTS / "raw").glob("*_swap.csv"))
if n_swap == 0:
    print("no swap rows yet; skipping figures")
else:
    steps = {
        "figures+tables": [PY, "scripts/make_figures.py", "--in", RESULTS / "raw", "--out", RESULTS / "figs",
                           "--tables", RESULTS / "tables", "--calibration", RESULTS / "calibration.json"],
        "web summary": [PY, "scripts/export_web_results.py", "--raw", RESULTS / "raw", "--figs", RESULTS / "figs",
                        "--web", RESULTS / "web"],
    }
    if time.time() < DEADLINE + 30 * 60:      # qualitative needs the swapper + judge on the GPU; skip if far over budget
        steps["qualitative (private)"] = [PY, "scripts/make_qualitative.py", "--config", "configs/main.yaml",
                                          "--out", RESULTS / "figs"]
    for name, cmd in steps.items():
        try:
            rc, _ = sh(cmd, log=LOGS / "figures.log", check=False)
            FIG_STATUS[name] = "ok" if rc == 0 else f"failed (exit {rc})"
        except Exception as e:
            FIG_STATUS[name] = f"failed ({type(e).__name__}: {e})"
print(FIG_STATUS)
print("files:", sorted(p.name for p in (RESULTS / "figs").glob("*"))[:30] if (RESULTS / "figs").is_dir() else "none")

In [ ]:
# 8) Package + summary. Download: right sidebar -> Output (or the "Output" tab of the saved version) -> the zip below.
package(partial=False)
print(f"\nZIP: {ZIP_PATH}  {ZIP_PATH.stat().st_size / 1e6:.1f} MB   (shard {SHARD})")

print("\nrows per config (this session's results dir; cloak rows = (attack, source), swap rows = one swap each):")
for c in RUN_CONFIGS:
    print(f"  {c:15s} cloak {count_rows(RESULTS / 'raw' / f'{c}_cloak.csv'):6d}   swap {count_rows(RESULTS / 'raw' / f'{c}_swap.csv'):7d}")

print("\nremaining work (whole manifest, i.e. counting all shards):")
for c in RUN_CONFIGS:
    sh([PY, "scripts/remaining_work.py", "--config", f"configs/{c}.yaml", "--results", RESULTS / "raw"], check=False)

try:
    print("\nstatus per config:", json.dumps(status))
except NameError:
    pass
print("\nIf anything above says 'missing': that is normal when the budget ran out or another shard still has to run.")
print("Send back the zip above (private; contains real faces). Do not make this notebook or its output public.")